# QuantaHire — RAG-Based CV Matching (Notebook Version)

**Pipeline:** MinerU Parser → Knowledge Graph (LightRAG) → Vector Embeddings → Two-Stage Ranking → Agentic Rewrite Loop

This notebook is the original research version, rebuilt from the production backend.
It reads CVs and JDs from local files, runs the full pipeline, and prints evaluation metrics.

---
## SECTION 1 — Install Dependencies
> Run once. Restart runtime if prompted, then skip this cell.

In [ ]:
!pip install -q raganything[paddleocr] lightrag-hku openai python-dotenv \
    sentence-transformers pandas==2.2.2 scikit-learn tqdm scipy PyMuPDF \
    transformers tf-keras paddlepaddle python-docx

import torch
if torch.cuda.is_available():
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"✅ GPU: {torch.cuda.get_device_name(0)} ({vram:.1f} GB)")
else:
    print("⚠️  No GPU detected — embeddings will be slower on CPU")

---
## SECTION 2 — Imports

In [ ]:
import os, json, asyncio, warnings, re, time, shutil, tempfile, zipfile
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from pathlib import Path
from tqdm import tqdm
from typing import List, Dict, Optional
from scipy.stats import spearmanr, kendalltau

# Document reading
import fitz        # PyMuPDF
import docx        # python-docx

# RAG framework
from raganything import RAGAnything, RAGAnythingConfig
from lightrag.llm.openai import openai_complete_if_cache
from lightrag.utils import EmbeddingFunc

# Embeddings
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# LLM client
from openai import AsyncOpenAI

print("✅ All imports loaded.")

---
## SECTION 3 — Configuration
> **Edit only this section.** Set your API key, paths, and model choices here.

In [ ]:
# ── API & Model ──────────────────────────────────────────────────────────────
API_KEY          = "YOUR_OPENAI_API_KEY_HERE"   # ← paste your key
LLM_MODEL        = "gpt-4o"
EMBED_MODEL_NAME = "sentence-transformers/all-mpnet-base-v2"
EMBED_DIM        = 768

# ── Experiment label (used for folder naming) ─────────────────────────────
EXPERIMENT_NAME  = "gpt4o_mpnet"   # change when you change LLM / embeddings

# ── Ranking settings ──────────────────────────────────────────────────────
TOP_K_FOR_LLM     = 10
WEIGHT_SIM        = 0.3
WEIGHT_LLM        = 0.7
MAX_ROUNDS        = 3

# ── Build async client ────────────────────────────────────────────────────
_client = AsyncOpenAI(api_key=API_KEY)

test = await _client.chat.completions.create(
    model=LLM_MODEL,
    messages=[{"role": "user", "content": "Reply with OK only."}],
    max_tokens=10, temperature=0,
)
print(f"✅ Config ready | LLM: {LLM_MODEL} → '{test.choices[0].message.content.strip()}'")
print(f"   Embeddings: {EMBED_MODEL_NAME} ({EMBED_DIM}-dim)")

In [ ]:
# ── File paths ────────────────────────────────────────────────────────────
# Point these at your NEW dataset.
# CV_FOLDER : folder containing .pdf or .docx CV files
# JDS_CSV   : CSV with columns  id | job_description | job_title
# GT_CSV    : CSV with ground-truth relevance scores (rows = CVs, cols = JDs)

CV_FOLDER = "data/cvs"                 # ← update
JDS_CSV   = "data/5_vacancies.csv"     # ← update
GT_CSV    = "data/ground_truth.csv"    # ← update (or set to None to skip eval)

# ── Working directories ───────────────────────────────────────────────────
BASE_DIR    = Path("./working")
RAG_STORAGE = str(BASE_DIR / f"rag_storage_{EXPERIMENT_NAME}")
PARSED_OUT  = str(BASE_DIR / "artifacts" / "parsed_cvs")
RESULTS_DIR = str(BASE_DIR / "results")
EXPORT_DIR  = str(BASE_DIR / "exports")

for p in [Path(RAG_STORAGE), Path(PARSED_OUT), Path(RESULTS_DIR), Path(EXPORT_DIR)]:
    p.mkdir(parents=True, exist_ok=True)

print("✅ Paths ready")
print(f"   CVs:     {CV_FOLDER}")
print(f"   JDs:     {JDS_CSV}")
print(f"   GT:      {GT_CSV}")
print(f"   Storage: {RAG_STORAGE}")

---
## SECTION 4 — Load Data
> Reads CVs and JDs from disk into memory.

In [ ]:
def get_text(path: str) -> str:
    """Extract plain text from .docx or .pdf file."""
    try:
        if path.lower().endswith('.docx'):
            doc  = docx.Document(path)
            text = ' '.join(p.text for p in doc.paragraphs)
            for table in doc.tables:
                for row in table.rows:
                    for cell in row.cells:
                        text += ' ' + cell.text
            return text.strip()
        elif path.lower().endswith('.pdf'):
            with fitz.open(path) as doc:
                return ' '.join(
                    page.get_text("text", sort=True) for page in doc
                ).strip()
    except Exception as e:
        print(f"  ⚠️  get_text failed for {path}: {e}")
    return ''

# ── Load job descriptions ─────────────────────────────────────────────────
df_jds     = pd.read_csv(JDS_CSV)
jd_records = [
    {
        'jd_id':    f"jd_{row['id']:03d}",
        'jd_text':  row['job_description'],
        'category': row['job_title'],
    }
    for _, row in df_jds.iterrows()
]

# ── Load CVs ──────────────────────────────────────────────────────────────
cv_records = []
cv_files   = sorted(
    [p for p in Path(CV_FOLDER).iterdir() if p.suffix.lower() in {'.docx', '.pdf'}],
    key=lambda p: (
        int(re.search(r'\d+', p.stem).group()) if re.search(r'\d+', p.stem) else 9999,
        p.name.lower()
    )
)

for i, p in enumerate(cv_files, start=1):
    text = get_text(str(p))
    if not text:
        print(f"  ⚠️  Empty text: {p.name}")
    cv_records.append({
        'cv_id':             f"cv_{i:03d}",
        'category':          'General',
        'path':              str(p),
        'text':              text,
        'original_filename': p.name,
    })

print(f"✅ {len(cv_records)} CVs  |  {len(jd_records)} JDs loaded\n")
for cv in cv_records:
    status = "✅" if cv['text'] else "❌ EMPTY"
    print(f"  {status}  {cv['cv_id']}  {cv['original_filename']}")
print()
for jd in jd_records:
    print(f"  ✅  {jd['jd_id']}  {jd['category']}")

---
## SECTION 5 — LLM Setup

In [ ]:
FORMAT_INSTRUCTION = (
    "Follow the exact format shown. Do not skip fields. "
    "Write N/A if missing. Nothing outside the format."
)

async def llm_func(prompt, system_prompt=None, history_messages=None, **kwargs):
    sys_msg  = f"{system_prompt}\n\n{FORMAT_INSTRUCTION}" if system_prompt else FORMAT_INSTRUCTION
    messages = [{"role": "system", "content": sys_msg}]
    if history_messages:
        messages.extend(history_messages)
    messages.append({"role": "user", "content": prompt})
    try:
        resp = await _client.chat.completions.create(
            model=LLM_MODEL, messages=messages,
            temperature=0, max_tokens=kwargs.get('max_tokens', 1024),
        )
        return resp.choices[0].message.content
    except Exception as e:
        print(f"  ⚠️  LLM error ({LLM_MODEL}): {e}")
        return None

async def vision_func(prompt, system_prompt=None, history_messages=None, **kwargs):
    return await llm_func(prompt, system_prompt, history_messages, **kwargs)

test = await llm_func(prompt="Reply with OK only.")
print(f"✅ llm_func ready | {LLM_MODEL} → '{(test or '').strip()}'")

---
## SECTION 6 — Embeddings Setup

In [ ]:
print(f"Loading embedding model: {EMBED_MODEL_NAME}...")
_embed_model = SentenceTransformer(EMBED_MODEL_NAME)

async def hf_embedding_func(texts: List[str]) -> np.ndarray:
    return _embed_model.encode(texts, normalize_embeddings=True)

embedding_func = EmbeddingFunc(
    embedding_dim=EMBED_DIM,
    max_token_size=512,
    func=hf_embedding_func,
)

async def compute_similarity_score(text1: str, text2: str) -> float:
    if not text1 or not text2:
        return 0.0
    emb1 = await hf_embedding_func([text1])
    emb2 = await hf_embedding_func([text2])
    sim  = cosine_similarity(emb1, emb2)[0][0]
    return round(max(0.0, float(sim)) * 100, 1)

def hybrid_score(similarity: float, llm_total: float,
                 weight_sim: float = WEIGHT_SIM, weight_llm: float = WEIGHT_LLM) -> float:
    return round(weight_sim * similarity + weight_llm * llm_total, 1)

print(f"✅ Embedding model ready | Dim: {EMBED_DIM} | Formula: {WEIGHT_SIM}×sim + {WEIGHT_LLM}×LLM")

---
## SECTION 7 — RAG / Knowledge Graph Initialisation

In [ ]:
config = RAGAnythingConfig(
    working_dir               = RAG_STORAGE,
    parser                    = 'mineru',
    parse_method              = 'auto',
    enable_image_processing   = False,
    enable_table_processing   = True,
    enable_equation_processing= False,
    display_content_stats     = True,
)

rag = RAGAnything(
    config            = config,
    llm_model_func    = llm_func,
    vision_model_func = vision_func,
    embedding_func    = embedding_func,
)

print("✅ RAGAnything initialised | Parser: MinerU | Graph: LightRAG | Storage:", RAG_STORAGE)

In [ ]:
# ── Option A: Restore a pre-built index from a local backup folder ────────
# Set INDEX_BACKUP to the path of your saved rag_storage folder, or None to skip.
INDEX_BACKUP = None   # e.g. "/path/to/saved/rag_storage_gpt4o_mpnet"

if INDEX_BACKUP:
    src = Path(INDEX_BACKUP)
    dst = Path(RAG_STORAGE)
    if dst.exists() and any(dst.iterdir()):
        print(f"✅ rag_storage already present — skipping copy")
    elif src.exists():
        shutil.copytree(str(src), str(dst))
        print(f"✅ Restored index from: {src}")
    else:
        print(f"❌ Backup not found: {src} — run indexing below instead")
else:
    print("ℹ️  No backup path set. Run the indexing cell below if no index exists yet.")

In [ ]:
# ── Option B: Index all CVs from scratch (~20 min, one-time) ─────────────
# Uncomment `await index_all_cvs()` to run.

async def index_all_cvs():
    to_process = [cv for cv in cv_records if Path(cv['path']).exists()]
    print(f"Indexing {len(to_process)} CVs | Parser: MinerU | LLM: {LLM_MODEL}\n")
    start = time.time()
    for cv in tqdm(to_process, desc="Indexing"):
        for attempt in range(3):
            try:
                await rag.process_document_complete(
                    file_path=cv['path'], output_dir=PARSED_OUT,
                )
                print(f"  ✅ {cv['cv_id']}")
                await asyncio.sleep(1)
                break
            except Exception as e:
                print(f"  ⚠️  Retry {attempt+1}/3: {str(e)[:80]}")
                await asyncio.sleep(10 * (attempt + 1))
        else:
            print(f"  ❌ Failed after 3 attempts: {cv['cv_id']}")
    print(f"\n✅ Indexing complete in {(time.time()-start)/60:.1f} minutes.")

# await index_all_cvs()
print("Cell ready. Uncomment the last line to run indexing, or set INDEX_BACKUP above.")

---
## SECTION 8 — LLM Scoring Functions

In [ ]:
SCORE_SYSTEM = (
    "You are a professional HR that rates resumes. Generate a score on the scale 1–5 for each "
    "work experience match, skills match, educational background match and certifications/extracurricular "
    "match based on the job description summary and resume. Additionally provide the reasons for the "
    "generated rating. Be strict in rating.\n\n"
    "The format of the output should be exactly like following:\n\n"
    "Rating: \n"
    "Work Experience Match: \n"
    "Skills Match: \n"
    "Educational Background Match\n"
    "Certifications/Extracurricular Match: \n\n"
    "Reasons for rating:\n"
)

def parse_rating(resp: str):
    scores  = {"work_exp": 3, "skills": 3, "education": 3, "certifications": 3}
    reasons = {"work_exp": "", "skills": "", "education": "", "certifications": ""}
    lines   = resp.strip().split("\n")
    in_rating, in_reasons = True, False
    reason_accum = []

    for line in lines:
        line = line.strip()
        if not line: continue
        if "Rating:" in line: continue
        if "Reasons for rating:" in line:
            in_rating, in_reasons = False, True
            continue
        if in_rating:
            for key, prefix in [("work_exp",       "Work Experience Match:"),
                                 ("skills",          "Skills Match:"),
                                 ("education",       "Educational Background Match"),
                                 ("certifications",  "Certifications/Extracurricular Match:")]:
                if line.startswith(prefix):
                    try:
                        val = line.split(":", 1)[1].strip() if ":" in line else line.replace(prefix,"").strip()
                        scores[key] = int(val.split()[0]) if val else 3
                    except: pass
        elif in_reasons:
            reason_accum.append(line)

    full  = " ".join(reason_accum)
    parts = [p.strip() + "." for p in full.split(".") if p.strip()]
    for i, key in enumerate(["work_exp","skills","education","certifications"]):
        reasons[key] = parts[i] if i < len(parts) else (full[:100] if i==0 else "")
    return scores, reasons

def total_match_from_scores(scores: dict) -> float:
    avg   = sum(scores.values()) / 4.0
    total = (avg - 1) * 25
    return max(0, min(100, round(total, 1)))

async def llm_score(jd_text: str, cv_context: str):
    prompt = (
        f"Job description summary:\n{jd_text[:3000]}\n\n"
        f"Resume content:\n{(cv_context or '')[:5000]}"
    )
    try:
        resp            = await llm_func(prompt=prompt, system_prompt=SCORE_SYSTEM)
        scores, reasons = parse_rating(resp)
        total           = total_match_from_scores(scores)
        verdict = (
            f"WE:{scores['work_exp']} Sk:{scores['skills']} "
            f"Ed:{scores['education']} Cert:{scores['certifications']} "
            f"| {reasons['skills'][:60]}"
        )
        return total, verdict
    except Exception as e:
        print(f"  ⚠️  LLM scoring error: {e}")
        return 50, f"LLM error: {str(e)[:100]}"

print("✅ HR scoring prompt and llm_score defined.")

---
## SECTION 9 — Agentic Query Rewrite

In [ ]:
def build_query(cv: dict, jd_text: str) -> str:
    return (
        f"Skills and experience of '{cv['cv_id']}' in {cv['category']}. "
        f"Relevant to: {jd_text[:2000]}"
    )

REWRITE_SYSTEM = (
    "You are an expert HR recruiter. The search query did not find "
    "the right candidates according to the recruiter's feedback.\n"
    "The recruiter rates candidates on: work experience, skills, education, and certifications.\n"
    "Rewrite the query to better match what the recruiter wants, focusing on these four areas.\n"
    "Output ONLY the new search query, nothing else."
)

async def rewrite_query(jd_text: str, current_query: str, feedback: str, history=None) -> str:
    history_text = ""
    if history:
        history_text = "\nPREVIOUS ATTEMPTS:\n"
        for h in history:
            history_text += (
                f"  Round {h['round']}: Top was {h['top_cv']} "
                f"({h.get('top_category','?')}, score {h['top_score']})\n"
            )
            if h.get('feedback'):
                history_text += f"    Recruiter: {h['feedback']}\n"

    prompt = (
        f"JOB DESCRIPTION:\n{jd_text[:400]}\n\n"
        f"CURRENT QUERY:\n{current_query}\n\n"
        f"RECRUITER FEEDBACK:\n{feedback}\n"
        f"{history_text}\n"
        "Write an improved search query. Output ONLY the query."
    )
    try:
        result = await llm_func(prompt=prompt, system_prompt=REWRITE_SYSTEM)
        return result.strip() if result else current_query
    except Exception as e:
        print(f"  ⚠️  Query rewrite error: {e}")
        return current_query

print("✅ Query builder and agentic rewrite defined.")

---
## SECTION 10 — Two-Stage Ranking Pipeline

In [ ]:
async def rank_cvs_for_jd(jd: dict, query_override: dict = None) -> pd.DataFrame:
    """
    Stage A: Embed JD + all CVs → cosine similarity (fast, no LLM)
    Stage B: LLM deep scoring for top-K candidates
    Stage C: Hybrid score = 0.3×similarity + 0.7×LLM score
    """
    jd_text = jd['jd_text']
    rows    = []

    print(f"  📐 Computing similarity for {len(cv_records)} CVs...")
    jd_emb = await hf_embedding_func([jd_text])

    sim_scores = []
    for cv in cv_records:
        text = cv.get('text', '')
        if not text:
            sim_scores.append((cv, 0.0))
            continue
        cv_emb = await hf_embedding_func([text])
        sim    = cosine_similarity(jd_emb, cv_emb)[0][0]
        sim_scores.append((cv, round(max(0.0, float(sim)) * 100, 1)))

    sim_scores.sort(key=lambda x: x[1], reverse=True)
    top_k = sim_scores[:TOP_K_FOR_LLM]
    rest  = sim_scores[TOP_K_FOR_LLM:]

    print(f"  🤖 LLM scoring top {TOP_K_FOR_LLM}...")

    for cv, sim_score in top_k:
        query   = (query_override or {}).get(cv['cv_id'], build_query(cv, jd_text))
        cv_text = cv.get('text', '')

        try:
            rag_ctx = await rag.aquery(query=query, mode='hybrid', top_k=10)
            context = cv_text + ('\n\n' + rag_ctx if rag_ctx and len(rag_ctx.strip()) > 100 else '')
        except Exception:
            context = cv_text

        llm_total, verdict = await llm_score(jd_text, context)
        final              = hybrid_score(sim_score, llm_total)

        rows.append({
            'cv_id': cv['cv_id'], 'category': cv['category'],
            'similarity': sim_score, 'llm_total': llm_total,
            'final_score': final, 'score': final,
            'verdict': verdict, 'jd_id': jd['jd_id'],
        })

    for cv, sim_score in rest:
        rows.append({
            'cv_id': cv['cv_id'], 'category': cv['category'],
            'similarity': sim_score, 'llm_total': None,
            'final_score': sim_score, 'score': sim_score,
            'verdict': f'[Similarity only — ranked below top {TOP_K_FOR_LLM}]',
            'jd_id': jd['jd_id'],
        })

    df        = pd.DataFrame(rows).sort_values('final_score', ascending=False).reset_index(drop=True)
    df['rank'] = df.index + 1
    return df

print("✅ Two-stage ranking pipeline ready.")

---
## SECTION 11 — Human-in-the-Loop (Agentic Loop)

In [ ]:
async def human_in_the_loop_ranking(jd: dict, max_rounds: int = MAX_ROUNDS) -> dict:
    print(f"\n{'='*70}")
    print(f"  📋 JD: {jd['jd_id']} — {jd['category']}")
    print(f"{'='*70}")
    print(f"  {jd['jd_text'][:200]}...")

    query_override = None
    history        = []

    for round_num in range(1, max_rounds + 1):
        print(f"\n  ── Round {round_num}/{max_rounds} {'─'*45}")
        print("  ⏳ Ranking...")

        df = await rank_cvs_for_jd(jd, query_override=query_override)

        print(f"\n  {'Rank':<5} {'CV':<12} {'Category':<35} {'Score':>6}")
        print(f"  {'─'*60}")
        for _, row in df.head(5).iterrows():
            print(f"  #{int(row['rank']):<4} {row['cv_id']:<12} {row['category']:<35} {row['score']:>5}")
            print(f"        └─ {row['verdict'][:70]}")

        print("\n  All candidates:")
        for _, row in df.iterrows():
            print(f"    #{int(row['rank']):>2} {row['cv_id']:<12} {row['category']:<35} → {row['score']:>5}")

        print()
        print("  ┌──────────────────────────────────────────────────────┐")
        print("  │  Type 'yes'     → APPROVE this ranking               │")
        print("  │  Type feedback  → RE-RANK with improved query        │")
        print("  └──────────────────────────────────────────────────────┘")

        user_input = input("  ➤ Decision: ").strip()

        if user_input.lower() in ('yes', 'y', 'approve', 'ok', 'accept', ''):
            print(f"\n  ✅ APPROVED at Round {round_num}!")
            history.append({
                'round': round_num, 'top_cv': df.iloc[0]['cv_id'],
                'top_category': df.iloc[0]['category'],
                'top_score': int(df.iloc[0]['score']),
                'approved': True, 'feedback': None, 'df': df.copy()
            })
            break
        else:
            feedback = user_input
            history.append({
                'round': round_num, 'top_cv': df.iloc[0]['cv_id'],
                'top_category': df.iloc[0]['category'],
                'top_score': int(df.iloc[0]['score']),
                'approved': False, 'feedback': feedback, 'df': df.copy()
            })
            print(f'\n  📝 Feedback: "{feedback[:80]}"')
            if round_num < max_rounds:
                print("  🔄 Rewriting retrieval query...")
                base_q = build_query({'cv_id': 'ideal', 'category': jd['category']}, jd['jd_text'])
                new_q  = await rewrite_query(jd['jd_text'], base_q, feedback, history)
                print(f"     New query: {new_q[:120]}")
                query_override = {cv['cv_id']: new_q for cv in cv_records}
            else:
                print("  ⚠️  Max rounds reached — using last ranking.")

    return {
        'jd_id':    jd['jd_id'],
        'category': jd['category'],
        'rounds':   len(history),
        'history':  history,
        'final_df': history[-1]['df'],
        'approved': history[-1].get('approved', False),
    }

print("✅ Human-in-the-loop function ready.")

---
## SECTION 12 — Run the Pipeline
> Processes all job descriptions one by one. Type `yes` to approve or give feedback to re-rank.

In [ ]:
all_results = []

for i, jd in enumerate(jd_records):
    print(f"\n{'━'*70}")
    print(f"  JD {i+1}/{len(jd_records)}  —  {jd['jd_id']}  —  {jd['category']}")
    print(f"{'━'*70}")
    result = await human_in_the_loop_ranking(jd, max_rounds=MAX_ROUNDS)
    all_results.append(result)

print(f"\n{'='*70}")
print(f"  ✅ Pipeline complete  —  {len(all_results)} JDs processed")
print(f"{'='*70}")

---
## SECTION 13 — Evaluation Against Ground Truth
> Computes NDCG, Precision@K, MRR, Recall@K, Spearman, and Kendall τ.
> **Skip this section if you do not have a ground-truth file.**

In [ ]:
# Load ground truth
# Expected format: rows = CVs (1–N), columns = one per JD, values = relevance scores (1–5, lower = more relevant)
if GT_CSV and Path(GT_CSV).exists():
    df_avg       = pd.read_csv(GT_CSV)
    num_jds      = len(jd_records)
    GROUND_TRUTH = df_avg.iloc[:, 1:num_jds+1].values.tolist()
    print(f"✅ Ground truth loaded: {len(GROUND_TRUTH)} CVs × {num_jds} JDs")
else:
    GROUND_TRUTH = None
    print("⚠️  No ground truth file — skipping evaluation")

cv_number_to_id = {i: f"cv_{i:03d}" for i in range(1, len(cv_records)+1)}

def dcg_at_k(scores, k):
    scores = np.array(scores[:k])
    return np.sum((2**scores - 1) / np.log2(np.arange(2, len(scores)+2)))

def ndcg_at_k(system_scores, ideal_scores, k):
    dcg   = dcg_at_k(system_scores, k)
    ideal = dcg_at_k(sorted(ideal_scores, reverse=True), k)
    return dcg / ideal if ideal > 0 else 0.0

def precision_at_k(system_top_k, relevant, k):
    return len(set(system_top_k[:k]) & set(relevant)) / k

def recall_at_k(system_top_k, relevant, k):
    return len(set(system_top_k[:k]) & set(relevant)) / len(relevant) if relevant else 0

def mrr(system_ranked, relevant):
    for i, cv in enumerate(system_ranked):
        if cv in relevant: return 1.0 / (i + 1)
    return 0.0

print("✅ Metric functions defined.")

In [ ]:
def evaluate_all_results(results):
    if GROUND_TRUTH is None:
        print("⚠️  No ground truth — cannot evaluate.")
        return

    print(f"\n{'='*70}")
    print("  📊 EVALUATION vs GROUND TRUTH")
    print(f"{'='*70}")

    jd_ids  = [jd['jd_id'] for jd in jd_records]
    metrics = {k: [] for k in ['spearman','kendall',
                                'ndcg3','ndcg5','ndcg10',
                                'p1','p3','p5','mrr','recall3','recall5']}

    for r in results:
        jd_idx = jd_ids.index(r['jd_id'])
        df     = r['final_df']

        system_ranks, gt_ranks = [], []
        for _, row in df.iterrows():
            cv_num = next((n for n,cid in cv_number_to_id.items() if cid==row['cv_id']), None)
            if cv_num and 1 <= cv_num <= len(GROUND_TRUTH):
                system_ranks.append(row['rank'])
                gt_ranks.append(GROUND_TRUTH[cv_num-1][jd_idx])

        if len(system_ranks) < 2: continue

        sp, _ = spearmanr(system_ranks, gt_ranks)
        kt, _ = kendalltau(system_ranks, gt_ranks)

        max_rank   = 5
        rel_scores = [max(0, max_rank+1 - gt) for gt in gt_ranks]
        sorted_p   = sorted(zip(system_ranks, rel_scores), key=lambda x: x[0])
        sys_ordered = [s for _,s in sorted_p]
        ideal       = sorted(rel_scores, reverse=True)

        relevant_cvs = [cv_number_to_id[n] for n in range(1, len(GROUND_TRUTH)+1)
                        if GROUND_TRUTH[n-1][jd_idx] <= 2]
        sys_cv_order = df.sort_values('rank')['cv_id'].tolist()

        ndcg3  = ndcg_at_k(sys_ordered, ideal, 3)
        ndcg5  = ndcg_at_k(sys_ordered, ideal, 5)
        ndcg10 = ndcg_at_k(sys_ordered, ideal, 10)
        p1     = precision_at_k(sys_cv_order, relevant_cvs, 1)
        p3     = precision_at_k(sys_cv_order, relevant_cvs, 3)
        p5     = precision_at_k(sys_cv_order, relevant_cvs, 5)
        mrr_s  = mrr(sys_cv_order, relevant_cvs)
        r3     = recall_at_k(sys_cv_order, relevant_cvs, 3)
        r5     = recall_at_k(sys_cv_order, relevant_cvs, 5)

        for key, val in [('spearman',sp),('kendall',kt),
                         ('ndcg3',ndcg3),('ndcg5',ndcg5),('ndcg10',ndcg10),
                         ('p1',p1),('p3',p3),('p5',p5),('mrr',mrr_s),
                         ('recall3',r3),('recall5',r5)]:
            metrics[key].append(val)

        print(f"\n  {r['jd_id']} ({r['category']}):")
        print(f"    Spearman: {sp:.3f}  Kendall: {kt:.3f}")
        print(f"    NDCG@3/5/10:     {ndcg3:.3f} / {ndcg5:.3f} / {ndcg10:.3f}")
        print(f"    Precision@1/3/5: {p1:.3f} / {p3:.3f} / {p5:.3f}")
        print(f"    MRR:             {mrr_s:.3f}")
        print(f"    Recall@3/5:      {r3:.3f} / {r5:.3f}")

    n = len(metrics['ndcg5'])
    print(f"\n  {'='*60}")
    print(f"  OVERALL AVERAGES (across {n} JDs)")
    print(f"  {'='*60}")
    print(f"  {'Metric':<25} {'Score':>8}  Interpretation")
    print(f"  {'─'*60}")
    for label, key, interp in [
        ("Spearman",    "spearman",  "rank correlation with human order"),
        ("Kendall τ",   "kendall",   "rank correlation with human order"),
        ("NDCG@3",      "ndcg3",     "top-3 quality (0–1, 1=perfect)"),
        ("NDCG@5",      "ndcg5",     "top-5 quality (0–1, 1=perfect)"),
        ("NDCG@10",     "ndcg10",    "top-10 quality (0–1, 1=perfect)"),
        ("Precision@1", "p1",        "most relevant CV at rank 1"),
        ("Precision@3", "p3",        "relevant CVs in top 3"),
        ("Precision@5", "p5",        "relevant CVs in top 5"),
        ("MRR",         "mrr",       "mean reciprocal rank (0–1)"),
        ("Recall@3",    "recall3",   "% relevant CVs found in top 3"),
        ("Recall@5",    "recall5",   "% relevant CVs found in top 5"),
    ]:
        print(f"  {label:<25} {np.mean(metrics[key]):>8.3f}  {interp}")

evaluate_all_results(all_results)

---
## SECTION 14 — Save Results to CSV

In [ ]:
save_path = Path(RESULTS_DIR) / "matching_results.csv"

all_rows = []
for r in all_results:
    for _, row in r['final_df'].iterrows():
        d = row.to_dict()
        d['rounds_needed'] = r['rounds']
        d['approved']      = r['approved']
        d['jd_category']   = r['category']
        all_rows.append(d)

final_df = pd.DataFrame(all_rows)
final_df.to_csv(save_path, index=False)

print(f"✅ Results saved → {save_path}")
print(f"   Rows: {len(final_df)}  |  Columns: {list(final_df.columns)}")
print()
print(final_df.head(10).to_string(index=False))

# If running in Colab, uncomment to download:
# from google.colab import files as colab_files
# colab_files.download(str(save_path))